# RoBERTa Experimental Smoke Test (1 Epoch)
This notebook is used to mathematically verify the HuggingFace `Trainer` pipeline, the BPE Tokenization mapping, and the Apple MPS (`macOS`) hardware acceleration logic on a small subset (500 samples) of the training data. 

This is an engineering check meant to ensure everything compiles and runs successfully before deploying the full mutli-epoch training sequence against the full 14k corpus.

In [1]:
import pandas as pd
import sys, os
sys.path.append(os.path.abspath('../src'))
from models.roberta import RobertaModel
from evaluation import compute_classification_metrics


/Users/anubhavgupta/Desktop/attribution_analysis/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Load a tiny restricted subset of the dataset using standard pandas slicing
train_df = pd.read_pickle('../data/processed/train.pkl').head(500)
val_df = pd.read_pickle('../data/processed/val.pkl').head(200)
print(f'Training on exactly {len(train_df)} samples to test PyTorch hardware pipeline...')


Training on exactly 500 samples to test PyTorch hardware pipeline...


In [3]:
model = RobertaModel(model_dir='../models/roberta_test_run')
model.train(train_df, val_df, epochs=1, batch_size=16)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading roberta-base...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/Users/anubhavgupta/Desktop/attribution_analysis/src/models/roberta.py:65: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


/Users/anubhavgupta/Desktop/attribution_analysis/venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,F1
1,No log,0.633369,0.000000


Saving best model to ../models/roberta_test_run...
Training complete.


In [4]:
# Evaluate the critically under-trained model to show F1 collapse
val_preds, _ = model.predict(val_df['text'].tolist(), batch_size=16)
metrics = compute_classification_metrics(val_df['label'].tolist(), val_preds)
print(f'Smoke Test Metrics: {metrics}')


Smoke Test Metrics: {'accuracy': 0.68, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
